# Fraud Detection - Exploratory Data Analysis

This notebook demonstrates how to use the fraud_detector package for exploratory data analysis.

In [ ]:
# Auto-reload modules for development
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import fraud_detector modules
from fraud_detector.data.loader import DataLoader, split_data
from fraud_detector.features.preprocessor import FeaturePreprocessor, handle_missing_values
from fraud_detector.utils.logger import logger
from config.config import settings

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Configuration

Let's verify our configuration is loaded correctly:

In [ ]:
print(f"Environment: {settings.environment}")
print(f"Random Seed: {settings.random_seed}")
print(f"Model Type: {settings.model_type}")
print(f"MLflow Experiment: {settings.mlflow_experiment_name}")
print(f"\nProject Root: {settings.project_root}")
print(f"Data Directory: {settings.get_absolute_path(settings.data_dir)}")

## 2. Load Data

### Example: Create Sample Data

Since we don't have real data yet, let's create a sample dataset:

In [ ]:
# Create sample fraud detection data
np.random.seed(settings.random_seed)

n_samples = 10000
fraud_rate = 0.02

sample_data = pd.DataFrame({
    'transaction_id': range(n_samples),
    'amount': np.random.exponential(scale=50, size=n_samples),
    'merchant_category': np.random.choice(['retail', 'online', 'restaurant', 'gas'], n_samples),
    'transaction_hour': np.random.randint(0, 24, n_samples),
    'day_of_week': np.random.randint(0, 7, n_samples),
    'distance_from_home': np.random.exponential(scale=10, size=n_samples),
    'num_transactions_24h': np.random.poisson(lam=3, size=n_samples),
    'is_fraud': np.random.binomial(1, fraud_rate, n_samples)
})

# Make fraud transactions more suspicious
fraud_mask = sample_data['is_fraud'] == 1
sample_data.loc[fraud_mask, 'amount'] *= 2
sample_data.loc[fraud_mask, 'distance_from_home'] *= 3
sample_data.loc[fraud_mask, 'num_transactions_24h'] += 5

logger.info(f"Created sample dataset with {len(sample_data):,} transactions")
print(f"\nDataset shape: {sample_data.shape}")
print(f"Fraud rate: {sample_data['is_fraud'].mean():.2%}")

In [ ]:
# Display first few rows
sample_data.head(10)

## 3. Data Exploration

In [ ]:
# Get data info using DataLoader
loader = DataLoader()
data_info = loader.get_data_info(sample_data)

print(f"Number of rows: {data_info['n_rows']:,}")
print(f"Number of columns: {data_info['n_columns']}")
print(f"Memory usage: {data_info['memory_usage_mb']:.2f} MB")
print(f"\nData types:\n{pd.Series(data_info['dtypes'])}")

In [ ]:
# Basic statistics
sample_data.describe()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
fraud_counts = sample_data['is_fraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values, color=['green', 'red'])
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'], 
            autopct='%1.2f%%', startangle=90, colors=['green', 'red'])
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.show()

## 4. Feature Analysis

In [ ]:
# Analyze numeric features by fraud status
numeric_features = ['amount', 'distance_from_home', 'num_transactions_24h', 'transaction_hour']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(numeric_features):
    sample_data.boxplot(column=feature, by='is_fraud', ax=axes[idx])
    axes[idx].set_title(f'{feature} by Fraud Status')
    axes[idx].set_xlabel('Is Fraud')
    axes[idx].set_ylabel(feature)

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation = sample_data[numeric_features + ['is_fraud']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Data Preprocessing

In [ ]:
# Prepare features and target
feature_cols = ['amount', 'merchant_category', 'transaction_hour', 
                'day_of_week', 'distance_from_home', 'num_transactions_24h']
target_col = 'is_fraud'

X = sample_data[feature_cols]
y = sample_data[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split data using the package function
train_df, val_df, test_df = split_data(
    sample_data,
    target_col='is_fraud',
    stratify=True
)

print(f"\nSplit sizes:")
print(f"Train: {len(train_df):,} ({len(train_df)/len(sample_data):.1%})")
print(f"Val: {len(val_df):,} ({len(val_df)/len(sample_data):.1%})")
print(f"Test: {len(test_df):,} ({len(test_df)/len(sample_data):.1%})")

In [ ]:
# Initialize preprocessor
numeric_features_list = ['amount', 'transaction_hour', 'day_of_week', 
                         'distance_from_home', 'num_transactions_24h']
categorical_features_list = ['merchant_category']

preprocessor = FeaturePreprocessor(
    numeric_features=numeric_features_list,
    categorical_features=categorical_features_list,
    scaler_type='robust'
)

# Fit on training data
X_train = train_df[feature_cols]
X_train_processed = preprocessor.fit_transform(X_train)

print(f"\nProcessed features shape: {X_train_processed.shape}")
print(f"Feature names: {preprocessor.get_feature_names()}")

## 6. Save Processed Data

In [ ]:
# Save to processed data directory
processed_path = settings.get_absolute_path(settings.data_dir / 'processed')

DataLoader.save_parquet(train_df, processed_path / 'train.parquet')
DataLoader.save_parquet(val_df, processed_path / 'val.parquet')
DataLoader.save_parquet(test_df, processed_path / 'test.parquet')

logger.info("Processed data saved successfully")

## Next Steps

1. **Feature Engineering**: Create additional features in `02_feature_engineering.ipynb`
2. **Model Training**: Train models in `03_model_training.ipynb`
3. **Model Evaluation**: Evaluate performance in `04_model_evaluation.ipynb`

Check the logs directory for detailed execution logs!